# Segmentation Analysis: Do Device, Channel, and Time Change Conversion?
Business question: does the tunnel behave differently across device, traffic sources, or months, and are the differences statistically meaningful?
Data: Synthetic Web Analytics Funnel Events (GenerateRealData, CC BY 4.0).

In [5]:
#IMPORT DAN KONEKSI
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

con = sqlite3.connect('../data/web_funnel_events.sqlite')

In [17]:
#SATU BARU PER SESI
sessions = pd.read_sql("SELECT * FROM sessions", con, parse_dates=["started_at"])

stage_flags = pd.read_sql("""
SELECT session_id,
  MAX(event_name='view_product')   AS viewed,
  MAX(event_name='add_to_cart')    AS carted,
  MAX(event_name='begin_checkout') AS checkout,
  MAX(event_name='purchase')       AS purchased
FROM events
GROUP BY session_id
""", con)

df = sessions.merge(stage_flags, on="session_id", how="left")
stage_cols = ["viewed", "carted", "checkout", "purchased"]
df[stage_cols] = df[stage_cols].fillna(0).astype(int)
df["month"] = df["started_at"].dt.to_period("M").astype(str)

print(df.shape, "overall conversion:", round(df["purchased"].mean() * 100, 2), "%")

(84000, 13) overall conversion: 9.06 %


In [18]:
#FUNGSI FUNNEL PER SEGMEN
def segment_funnel(df, col):
    g = df.groupby(col).agg(
        sessions=("session_id", "count"),
        viewed=("viewed", "sum"),
        carted=("carted", "sum"),
        checkout=("checkout", "sum"),
        purchased=("purchased", "sum"))
    out = pd.DataFrame({
        "sessions": g["sessions"],
        "view_rate_%": g["viewed"] / g["sessions"] * 100,
        "cart_rate_%": g["carted"] / g["viewed"] * 100,
        "checkout_rate_%": g["checkout"] / g["carted"] * 100,
        "purchase_rate_%": g["purchased"] / g["checkout"] * 100,
        "overall_conversion_%": g["purchased"] / g["sessions"] * 100
    }).round(2)
    return out.sort_values("overall_conversion_%", ascending=False)

display(segment_funnel(df, "device"))
display(segment_funnel(df, "referrer"))
display(segment_funnel(df, "month"))

,sessions,view_rate_%,cart_rate_%,checkout_rate_%,purchase_rate_%,overall_conversion_%
device,,,,,,
mobile,48859,67.17,40.19,56.56,59.93,9.15
desktop,28528,67.74,39.97,56.85,58.01,8.93
tablet,6613,67.10,39.44,56.69,59.48,8.92


,sessions,view_rate_%,cart_rate_%,checkout_rate_%,purchase_rate_%,overall_conversion_%
referrer,,,,,,
partner site,4252,66.86,40.10,58.68,60.54,9.52
instagram,9305,67.53,40.36,56.55,60.32,9.30
facebook,7546,67.86,41.48,56.07,58.61,9.25
bing,4091,67.25,40.86,55.34,60.61,9.22
direct,19967,67.24,39.23,56.80,60.11,9.00
google,32182,67.34,40.34,56.90,58.05,8.97
newsletter,6657,67.34,38.63,55.54,60.08,8.68


,sessions,view_rate_%,cart_rate_%,checkout_rate_%,purchase_rate_%,overall_conversion_%
month,,,,,,
2026-09,26697,67.49,40.44,57.16,59.90,9.35
2026-07,28628,67.19,40.03,57.04,58.45,8.97
2026-08,28675,67.40,39.73,55.83,59.41,8.88
